## 1. Protein Structural Change Data Base (PSCDB)

The [Protein Structural Change DataBase](https://academic.oup.com/nar/article/40/D1/D554/2903618) provides information about the structural changes observed in proteins upon ligand binding.

Let us look at one example from PSCDB: __Adenylate kinase__.

Adenylate kinase is a useful case study because it undergoes a clear conformational change between its open, ligand-free state and its closed, ligand-bound state. We will compare two crystal structures from the [Protein Databank (PDB)](https://www.ebi.ac.uk/pdbe/):

- **4AKE**: ligand-free adenylate kinase
- **2ECK**: ligand-bound adenylate kinase

By comparing these structures, we can start to see how protein motion can be represented using structural data and graph-based methods.

In [ ]:
GNN_DATA_FOLDER = "/projects/nb4170/gnn_data"

from src.utils import display_two_protein_graphs_overlay
display_two_protein_graphs_overlay(
    os.path.join(GNN_DATA_FOLDER, "4AKE.pdb"),
    os.path.join(GNN_DATA_FOLDER, "2ECK.pdb"),
    labels=("4AKE", "2ECK"),)

The ligand-induced structural changes in the PSCB database were classified into seven classes: 

```
1. Coupled Domain Motion (N=59)
2. Independent Domain Motion (N=70)
3. Coupled Local Motion (N=125)
4. Independent Local Motion (N=135)
5. Burying Ligand Motion (N=104)
6. No significant motion (N=311)
7. Other types (N=35)
```

Here, “coupled motion” refers to cases where both parts contributing to a structural movement contact the ligand in the ligand-bound structure. If only one part, or neither part, contacts the ligand, the motion is classified as “independent” or “uncoupled.” The following image illustrates the different classes of structural change:

![Protein Structural Change Classes](./image/pscdb_7.jpg)

In [ ]:
# Downloading the data for the GNN class if you are not using DelftBlue. Uncomment the next two lines
#!git clone https://github.com/courtel/GNN_dl_workshop.git gnn_data

# Point this to the appropriate gnn_data folder
#GNN_DATA_FOLDER = "/projects/nb4170/gnn_data" 


Let us now have a closer look at the database. Run the following cell to load the data and explore its structure. 

In [ ]:
import os
import pandas as pd

GNN_DATA_FOLDER = "./gnn_data/"
pscdb_csv_path = os.path.join(GNN_DATA_FOLDER, "df_pscdb.csv")

df_pscdb = pd.read_csv(pscdb_csv_path)
print(df_pscdb.head())

The important columns for our analysis are:

- `free_pdb`: the unique accession code for the ligand-free protein structure deposited in the Protein Data Bank, also known as the PDB ID.

- `motion_type`: the class label describing the type of structural change observed in the protein upon ligand binding.



## 2. Data pre-processing

Before we can use the dataset for machine learning, we need to convert the raw table into a format that models can work with.

### 2a) Encode class labels and link them to PDB IDs

The column `motion_type` contains the class label describing the type of structural change. We then map these numerical labels to the corresponding PDB structure given in the `free_pdb` column.

In [ ]:
mapping_pdb_to_motion_type = {x["free_pdb"]:x["motion_type"] for _, x in df_pscdb.iterrows()}


In [ ]:
from src.utils import show_random_elements_in_dictionary
show_random_elements_in_dictionary(mapping_pdb_to_motion_type, 10)

For this task, we can restrict ourselves to domain motions as they can be easily captured using residue-level abstractions. We will only consider "independent_domain_motion", "coupled_domain_motion", and "no_significant_motion" classes.

In [ ]:
included_motion_types = ["independent_domain_motion", "independent_local_motion", "no_significant_motion"]
filtered_mapping_pdb_to_motion_type = {
                                    k:v for k,v in mapping_pdb_to_motion_type.items()\
                                    if v in included_motion_types
                                }

original_num_samples = len(mapping_pdb_to_motion_type)
filtered_num_samples = len(filtered_mapping_pdb_to_motion_type)
print(f"Original number of samples: {original_num_samples}")
print(f"Filtered number of samples: {filtered_num_samples}")
print(f"Random samples from the filtered mapping:")
show_random_elements_in_dictionary(filtered_mapping_pdb_to_motion_type, 10)

### 2.2 Convert motion type into integer strings

Since machine learning models typically expect numerical labels, we first convert each motion class into an integer. 


In [ ]:
motion_type_to_integer = {x : i for i, x in enumerate(set(filtered_mapping_pdb_to_motion_type.values()))}
for k, v in motion_type_to_integer.items():
    count = sum(1 for motion_type in filtered_mapping_pdb_to_motion_type.values() if motion_type == k)
    print(f"{k} : {v} (count: {count})")

pdb_id_to_integer_label = {k: motion_type_to_integer[v] for k, v in filtered_mapping_pdb_to_motion_type.items()}
print(f"Random samples from the pdb_id to integer label mapping:")
show_random_elements_in_dictionary(pdb_id_to_integer_label, 10)
    

### 2.3 Split the data into test, train and validation sets

Next, we divide the dataset into **training**, **validation**, and **test** sets.

As a reminder: 

- The **training set** is used to fit the model.
- The **validation set** is used to tune model choices and monitor performance during training.
- The **test set** is kept separate and used only at the end to estimate how well the model generalises to unseen proteins.

For each PDB ID, we will use the `construct_graph` function from the first module to create a graph representation of the corresponding protein structure.

The PDB files are stored in the `gnn_data` directory, with the following structure:

```
gnn_data/
├── all_datasets/
│   ├── test/
│   │   ├── 1a3h.pdb
│   │   ├── 1agj.pdb
│   │   └── ...
│   ├── train/
│   │   ├── 1a48.pdb
│   │   ├── 1ayf.pdb
│   │   └── ...
│   └── val/
│       ├── 1bs0.pdb
│       ├── 1cmb.pdb
│       └── ...
```

In [ ]:
raw_pdb_files_test = [\
    f for f in os.listdir(os.path.join(GNN_DATA_FOLDER, "all_datasets", "test"))\
        if f.endswith(".pdb")
]
raw_pdb_files_train = [\
    f for f in os.listdir(os.path.join(GNN_DATA_FOLDER, "all_datasets", "train"))\
        if f.endswith(".pdb")
]
raw_pdb_files_val = [\
    f for f in os.listdir(os.path.join(GNN_DATA_FOLDER, "all_datasets", "val"))\
        if f.endswith(".pdb")
]

## Curate the raw PDB files to only include those that are in the filtered mapping
pdb_files_test = [f for f in raw_pdb_files_test if f[:-4] in filtered_mapping_pdb_to_motion_type]
pdb_files_train = [f for f in raw_pdb_files_train if f[:-4] in filtered_mapping_pdb_to_motion_type]
pdb_files_val = [f for f in raw_pdb_files_val if f[:-4] in filtered_mapping_pdb_to_motion_type]

# Convert the pdb paths to full paths
pdb_files_test = [os.path.join(GNN_DATA_FOLDER, "all_datasets", "test", f) for f in pdb_files_test]
pdb_files_train = [os.path.join(GNN_DATA_FOLDER, "all_datasets", "train", f) for f in pdb_files_train]
pdb_files_val = [os.path.join(GNN_DATA_FOLDER, "all_datasets", "val", f) for f in pdb_files_val]

print(f"Number of raw PDB files in test set: {len(pdb_files_test)}")
print(f"Number of raw PDB files in train set: {len(pdb_files_train)}")
print(f"Number of raw PDB files in val set: {len(pdb_files_val)}")

It is a good time to check how the classes are balanced between the train, test, and validation sets. 

In [ ]:
# Check the distribution of motion types in the filtered dataset between the train, test, and val sets
def count_motion_types_in_pdb_files(pdb_files, mapping_pdb_to_motion_type):
    motion_type_counts = {motion_type: 0 for motion_type in set(mapping_pdb_to_motion_type.values())}
    for pdb_file in pdb_files:
        pdb_id = os.path.basename(pdb_file)[:-4]  # Remove the .pdb extension
        if pdb_id in mapping_pdb_to_motion_type:
            motion_type = mapping_pdb_to_motion_type[pdb_id]
            motion_type_counts[motion_type] += 1
    motion_type_counts["total"] = sum(motion_type_counts.values())
    return motion_type_counts



test_motion_type_counts = count_motion_types_in_pdb_files(
    pdb_files_test, filtered_mapping_pdb_to_motion_type
)
train_motion_type_counts = count_motion_types_in_pdb_files(
    pdb_files_train, filtered_mapping_pdb_to_motion_type
)
val_motion_type_counts = count_motion_types_in_pdb_files(
    pdb_files_val, filtered_mapping_pdb_to_motion_type
)
print("Motion type distribution in test set:")
for motion_type, count in test_motion_type_counts.items():
    print(f"{motion_type}: {count} : {count/test_motion_type_counts['total']:.2%}")
print("\nMotion type distribution in train set:")
for motion_type, count in train_motion_type_counts.items():
    print(f"{motion_type}: {count} : {count/train_motion_type_counts['total']:.2%}")
print("\nMotion type distribution in val set:")
for motion_type, count in val_motion_type_counts.items():
    print(f"{motion_type}: {count} : {count/val_motion_type_counts['total']:.2%}")



### 2.4 Create a protein graph for each PDB ID

Next, we convert each protein structure into a graph. For each PDB ID in the filtered dataset, we load the corresponding PDB file and use `construct_graph` to create a protein graph. 

In this graph representation:

- **nodes** represent amino acid residues
- **edges** represent structural relationships between residuess

In [ ]:
import networkx as nx
def display_graph(protein_graph):
    from plotly import graph_objects as go
    # Convert to NetworkX graph
    nx_graph = nx.Graph(protein_graph)
    pos_2d = nx.spring_layout(nx_graph, seed=0)

    # Build edge traces for the 2D graph
    edge_x, edge_y = [], []
    for edge in nx_graph.edges():
        x0, y0 = pos_2d[edge[0]]
        x1, y1 = pos_2d[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=1, color='gray'),
        hoverinfo='all',
        mode='lines'
    )

    # Build node trace for the 2D graph
    node_x = [pos_2d[node][0] for node in nx_graph.nodes()]
    node_y = [pos_2d[node][1] for node in nx_graph.nodes()]
    node_labels = list(nx_graph.nodes())

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',           # was 'markers'
        text=node_labels,              # text shown next to each node
        textposition='top center',     # where the label sits relative to marker
        hovertext=node_labels,         # text shown on hover
        hoverinfo='text',
        marker=dict(size=15, color='lightblue', line=dict(width=1, color='black')),
        textfont=dict(size=8),
    )

    # Create the figure
    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_scenes(
        xaxis=dict(showgrid=False, showticklabels=False, showbackground=False, title=''),
        yaxis=dict(showgrid=False, showticklabels=False, showbackground=False, title=''),    
    )
    fig.show()
    


In [ ]:
from src.utils import convert_pdb_path_to_graph, display_protein_with_graph
import graphein.protein as gp
pdb_path_example = pdb_files_train[0]
print(f"Example PDB path: {pdb_path_example}")

protein_graph_example = convert_pdb_path_to_graph(pdb_path_example, distance_threshold=6, contain_b_factor=True)
print(f"Converted PDB file into graph with {protein_graph_example.number_of_nodes()} nodes")



In [ ]:
example_node = list(protein_graph_example.nodes())[2]
print(f"Example node: {example_node}")
example_node_attributes = protein_graph_example.nodes[example_node]
print(f"Attributes of example node:")
for attr, value in example_node_attributes.items():
    if isinstance(value, (int, float, str)):
        print(f"\t {attr}: {value}")
    else:
        print(f"\t {attr}: (type: {type(value)}, length: {len(value)})")


example_edge = list(protein_graph_example.edges())[2]
print(f"Example edge: {example_edge}")
example_edge_attributes = protein_graph_example.edges[example_edge]
print(f"Attributes of example edge:")
for attr, value in example_edge_attributes.items():
    if isinstance(value, (int, float, str)):
        print(f"\t {attr}: {value}")
    else:
        print(f"\t {attr}: (type: {type(value)}, length: {len(value)})")
        

### 2.5 Convert graphs to PyTorch Geometric `Data` objects

We now convert each `networkx.Graph` into a `torch_geometric.data.Data` object.

This is the standard graph format used by **PyTorch Geometric**. A `Data` object stores the information needed for graph neural network training, such as:

- `x`: node features
- `edge_index`: graph connectivity
- `edge_attr`: edge features, if available
- `y`: the target label

After this step, each protein graph is represented in a format that can be passed directly to a graph neural network.

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.utils import from_networkx
from src.utils import set_seed

set_seed(1)
tensor_graph_example = from_networkx(protein_graph_example)


Explore the tensor data structure

In [ ]:
# Convert the node attributes to tensors
def convert_pdb_path_to_pytorch_tensor(pdb_path, pdb_id_to_integer_label):
    # Convert the PDB file to a graph
    protein_graph = convert_pdb_path_to_graph(pdb_path, distance_threshold=6, contain_b_factor=True)
    # Convert the graph to a PyTorch Geometric Data object
    data = from_networkx(protein_graph)

    
    # Add name
    data.name = os.path.basename(pdb_path)[:-4]  # Remove the .pdb extension

    # Add target
    data.y = pdb_id_to_integer_label[data.name]
    
    return data

In [ ]:
tensor_graph_example = convert_pdb_path_to_pytorch_tensor(pdb_path_example, pdb_id_to_integer_label)
tensor_graph_example

In [ ]:
from tqdm import tqdm
def get_list_of_pytorch_graphs_from_pdb_paths(pdb_paths, pdb_id_to_integer_label):
    pytorch_graphs = []
    for pdb_path in tqdm(pdb_paths, desc="Converting PDB files to PyTorch tensors"):
        pytorch_graph = convert_pdb_path_to_pytorch_tensor(pdb_path, pdb_id_to_integer_label)
        pytorch_graphs.append(pytorch_graph)
    return pytorch_graphs


In [ ]:
train_pytorch_graphs = get_list_of_pytorch_graphs_from_pdb_paths(pdb_files_train, pdb_id_to_integer_label)
test_pytorch_graphs = get_list_of_pytorch_graphs_from_pdb_paths(pdb_files_test, pdb_id_to_integer_label)
val_pytorch_graphs = get_list_of_pytorch_graphs_from_pdb_paths(pdb_files_val, pdb_id_to_integer_label)
print(f"Number of PyTorch graphs in train set: {len(train_pytorch_graphs)}")
print(f"Number of PyTorch graphs in test set: {len(test_pytorch_graphs)}")
print(f"Number of PyTorch graphs in val set: {len(val_pytorch_graphs)}")


In [ ]:
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool

train_dataloader = DataLoader(train_pytorch_graphs, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_pytorch_graphs, batch_size=32, shuffle=False)
val_dataloader = DataLoader(val_pytorch_graphs, batch_size=32, shuffle=False)


### 2.6 Training a Graph Convolutional Network

Now that the protein structures have been converted into PyTorch Geometric `Data` objects, we can train a **Graph Convolutional Network (GCN)** to classify protein structural changes.

A GCN learns by combining information from each node with information from its neighbouring nodes. This makes it well suited for protein graphs, where residues are connected through peptide bonds or spatial contacts.

We will use a simple model architecture:

- two graph convolutional layers
- a graph-level pooling step
- one fully connected layer for classification

The goal is to predict the `motion_type` label for each protein graph.


#### 2.6.1 Model definition

In [ ]:
## Useful functions
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_accuracy = 0
    for data in dataloader:
        # Move data to the same device as the model
        data = data.to(device)
        # Forward pass
        out = model(data)
        # Compute loss
        loss = criterion(out, data.y)
        total_loss += loss.item() 
        num_correct = (out.argmax(dim=1) == data.y).sum().item()
        total_accuracy += num_correct 

    avg_loss = total_loss / len(dataloader)
    avg_accuracy = total_accuracy / len(dataloader.dataset)   
    return avg_loss, avg_accuracy

def make_prediction_for_pdb(pdb_path, model, pdb_id_to_integer_label, device):
    model.eval()
    data = convert_pdb_path_to_pytorch_tensor(pdb_path, pdb_id_to_integer_label)
    data = data.to(device)
    with torch.no_grad():
        out = model(data)
        predicted_label = out.argmax(dim=1).item()
    return predicted_label

def display_pdb_graph_with_prediction(pdb_path, model, pdb_id_to_integer_label, device):
    from src.utils import return_protein_graph_for_pdb_path, display_protein_with_graph
    predicted_label = make_prediction_for_pdb(pdb_path, model, pdb_id_to_integer_label, device)
    predicted_motion_type = [k for k, v in motion_type_to_integer.items() if v == predicted_label][0]
    print(f"Predicted motion type for {os.path.basename(pdb_path)}: {predicted_motion_type}")
    protein_graph = return_protein_graph_for_pdb_path(pdb_path)
    display_protein_with_graph(protein_graph)

In [ ]:
class GCNGraphLevel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCNGraphLevel, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)

        x = global_max_pool(x, batch)  

        out = self.lin(x)

        return out

#### 2.6.2 Set hyperparameters

In [ ]:
NUM_EPOCHS = 100
LEARNING_RATE = 0.01
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#### 2.6.3 Train the model

In [ ]:
model = GCNGraphLevel(in_channels=4, hidden_channels=16, out_channels=len(motion_type_to_integer))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.CrossEntropyLoss()

model.to(device)

In [ ]:
train_losses = []
val_losses = []
train_accuracy = []
val_accuracy = []
print(f"Epoch \t Train Loss \t Train Accuracy \t Val Loss \t Val Accuracy")
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for data in train_dataloader:
        # Move data to the same device as the model
        data = data.to(device)
        # Zero gradients
        optimizer.zero_grad()
        # Forward pass
        out = model(data)
        # Compute loss
        loss = criterion(out, data.y)
        # Backward pass 
        loss.backward()
        # Gradient descent 
        optimizer.step()
        # Accumulate loss
        total_loss += loss.item() 
    

    avg_train_loss, avg_train_accuracy = evaluate(model, train_dataloader, criterion, device)
    avg_val_loss, avg_val_accuracy = evaluate(model, val_dataloader, criterion, device)
    
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_accuracy.append(avg_train_accuracy)
    val_accuracy.append(avg_val_accuracy)

    print(f"{epoch+1}/{NUM_EPOCHS} \t {avg_train_loss:.4f} \t {avg_train_accuracy:.4f} \t {avg_val_loss:.4f} \t {avg_val_accuracy:.4f}")

#### 2.6.4 Evaluate the training

In [ ]:
# Plot the training and validation loss curves
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(train_losses, label='Train Loss')
ax[0].plot(val_losses, label='Val Loss')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss')
ax[0].set_title('Training and Validation Loss')
ax[0].legend()
ax[1].plot(train_accuracy, label='Train Accuracy')
ax[1].plot(val_accuracy, label='Val Accuracy')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Accuracy')
ax[1].set_title('Training and Validation Accuracy')
ax[1].legend()
plt.tight_layout()
plt.show()

### 2.7 Evaluating the model

In [ ]:
# Evaluate the model on the test set
_, test_accuracy = evaluate(model, test_dataloader, criterion, device)
print(f"Test Accuracy: {test_accuracy*100:.2f}%")


In [ ]:
test_pdb_path = pdb_files_test[1]
true_label = pdb_id_to_integer_label[os.path.basename(test_pdb_path)[:-4]]
true_motion_type = [k for k, v in motion_type_to_integer.items() if v == true_label][0]
print(f"True motion type for {os.path.basename(test_pdb_path)}: {true_motion_type}")
display_pdb_graph_with_prediction(test_pdb_path, model, pdb_id_to_integer_label, device)

With 3 predicted classes, a random baseline accuracy would be around 33.3%. Did your model perform better than that? 

Can you think of any ways to improve the model's performance?